In [2]:
import pandas as pd
import shutil
import os
from tqdm import tqdm

# ── Load files ───────────────────────────────────────────────────────────────
labelled = pd.read_csv("final_labelled.csv")
mapping  = pd.read_csv("gz2_filename_mapping.csv")

# ── Merge on objid = dr7objid ────────────────────────────────────────────────
merged = mapping.merge(
    labelled[["dr7objid", "morphology_label"]],
    left_on  = "objid",
    right_on = "dr7objid",
    how      = "inner"
)
print(f"Mapping file      : {len(mapping):>7,} rows")
print(f"Labelled file     : {len(labelled):>7,} rows")
print(f"Matched           : {len(merged):>7,} rows")

# ── Folder setup ─────────────────────────────────────────────────────────────
images_dir = "images"   # folder containing asset_id.jpg files

output_dirs = {
    0: "dataset/ellipticals",
    1: "dataset/spirals",
    2: "dataset/irregulars"
}

for path in output_dirs.values():
    os.makedirs(path, exist_ok=True)

# ── Copy images ──────────────────────────────────────────────────────────────
not_found = 0
copied    = {0: 0, 1: 0, 2: 0}

for _, row in tqdm(merged.iterrows(), total=len(merged), desc="Copying images"):
    src = os.path.join(images_dir, f"{row['asset_id']}.jpg")
    
    if not os.path.exists(src):
        not_found += 1
        continue
    
    label   = int(row["morphology_label"])
    dst_dir = output_dirs[label]
    shutil.copy2(src, os.path.join(dst_dir, f"{row['asset_id']}.jpg"))
    copied[label] += 1

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\nDataset created:")
print(f"  dataset/ellipticals/ : {copied[0]:>7,} images")
print(f"  dataset/spirals/     : {copied[1]:>7,} images")
print(f"  dataset/irregulars/  : {copied[2]:>7,} images")
print(f"  Images not found     : {not_found:>7,}")
print(f"  Total copied         : {sum(copied.values()):>7,}")

Mapping file      : 355,990 rows
Labelled file     :  31,374 rows
Matched           :  31,374 rows


Copying images: 100%|██████████| 31374/31374 [01:39<00:00, 315.00it/s]


Dataset created:
  dataset/ellipticals/ :  12,643 images
  dataset/spirals/     :  10,336 images
  dataset/irregulars/  :   8,395 images
  Images not found     :       0
  Total copied         :  31,374


In [3]:
import pandas as pd
import os

labelled = pd.read_csv("final_labelled.csv")
mapping  = pd.read_csv("gz2_filename_mapping.csv")
images_dir = "images"

# Find which asset_ids actually exist on disk
merged = mapping.merge(
    labelled[["dr7objid", "morphology_label"]],
    left_on="objid", right_on="dr7objid", how="inner"
)

# Check which images exist
merged["image_exists"] = merged["asset_id"].apply(
    lambda x: os.path.exists(os.path.join(images_dir, f"{x}.jpg"))
)

missing = merged[~merged["image_exists"]]
print(f"Missing images: {len(missing)}")
print(missing[["objid", "asset_id", "morphology_label"]])

# Get the dr7objids to drop
ids_to_drop = missing["dr7objid"].values

# Remove from labelled and save
labelled_clean = labelled[~labelled["dr7objid"].isin(ids_to_drop)]
labelled_clean.to_csv("final_labelled.csv", index=False)

print(f"\nBefore : {len(labelled):,}")
print(f"Removed: {len(ids_to_drop)}")
print(f"After  : {len(labelled_clean):,}")
print("final_labelled.csv updated.")

Missing images: 0
Empty DataFrame
Columns: [objid, asset_id, morphology_label]
Index: []

Before : 31,374
Removed: 0
After  : 31,374
final_labelled.csv updated.


In [4]:
labelled_clean.columns.tolist()

['specobjid_x',
 'dr8objid',
 'dr7objid',
 'ra',
 'dec',
 'rastring',
 'decstring',
 'sample',
 'gz2class',
 'total_classifications',
 'total_votes',
 't01_smooth_or_features_a01_smooth_count',
 't01_smooth_or_features_a01_smooth_weight',
 't01_smooth_or_features_a01_smooth_fraction',
 't01_smooth_or_features_a01_smooth_weighted_fraction',
 't01_smooth_or_features_a01_smooth_debiased',
 't01_smooth_or_features_a01_smooth_flag',
 't01_smooth_or_features_a02_features_or_disk_count',
 't01_smooth_or_features_a02_features_or_disk_weight',
 't01_smooth_or_features_a02_features_or_disk_fraction',
 't01_smooth_or_features_a02_features_or_disk_weighted_fraction',
 't01_smooth_or_features_a02_features_or_disk_debiased',
 't01_smooth_or_features_a02_features_or_disk_flag',
 't01_smooth_or_features_a03_star_or_artifact_count',
 't01_smooth_or_features_a03_star_or_artifact_weight',
 't01_smooth_or_features_a03_star_or_artifact_fraction',
 't01_smooth_or_features_a03_star_or_artifact_weighted_fract